In [ ]:
!nvidia-smi

In [ ]:
%%writefile relu.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void relu(float *x, float *y, int N) {
    int i = blockIdx.x*blockDim.x + threadIdx.x;
    if (i < N) y[i] = fmaxf(x[i], 0.f);
}

int main() {
    int N = 1<<24;
    float *dx, *dy;
    cudaMalloc(&dx, N*4); cudaMalloc(&dy, N*4);
    relu<<<(N+255)/256, 256>>>(dx, dy, N);
    cudaDeviceSynchronize();
    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    int reps = 100; cudaEventRecord(s);
    for (int i = 0; i < reps; i++) relu<<<(N+255)/256,256>>>(dx, dy, N);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms; cudaEventElapsedTime(&ms,s,e);
    printf("ReLU  %.2f GB/s\n", 2.0*N*4*reps/(ms*1e6));
    cudaFree(dx); cudaFree(dy);
}

In [ ]:
!nvcc -arch=sm_75 -O2 -o relu relu.cu && ./relu